# Part 2: Building a Conversational Chatbot

This guide provides instructions for extending your Large Language Model (LLM) application into a conversational chatbot. You should have already completed the setup in the first guide and have a working single-response LLM application.

> This notebook is a guide to enhancing your project. **Do not run the code cells here directly.** Follow the instructions in your local development environment.

-----

## 1.&nbsp; From a Single Answer to a Real Conversation

Before we write any code, it's important to understand a key conceptual leap we're about to make.

In the first part of our project, our application was stateless. We used the `llm.complete()` method, which took a single question and returned a single answer. It had no memory of past interactions. If you asked it a follow-up question, it would have no idea what you were talking about.

To build a true chatbot, we need a stateful system - one that can remember the conversation history. This is the primary job of a Chat Engine. A chat engine wraps the LLM and automatically manages the back-and-forth dialogue, giving the model the context it needs to hold a coherent conversation. We'll start with LlamaIndex's `SimpleChatEngine`.

-----

## 2.&nbsp; Update the Configuration with a System Prompt

A chatbot needs instructions on how to behave. We'll add a system prompt to our configuration file to define its personality.

1.  Open `src/config.py` in your VSCode editor.
2.  Comment out `LLM_QUESTION`, we no longer need it.
2.  Add the `LLM_SYSTEM_PROMPT` variable to the file. Your `config.py` should now look like this:

In [ ]:
# Example question, not used in the current code
# LLM_QUESTION: str = (
#     "What is the capital of France?"
# )
LLM_SYSTEM_PROMPT: str = (
    "You are a helpful chatbot. Be friendly and conversational."
)


You might be thinking, "Couldn't I just tell the user to start their chat with the instructions for the personality they would like?" While you could, a system prompt is fundamentally different and more powerful for two reasons:

1.  It's a Meta-Instruction: A system prompt operates at a level above the user's conversation. It sets the "rules of the game" for the AI, which the model is heavily incentivised to follow throughout the entire dialogue.
2.  It's Persistent: The chat engine ensures the system prompt is always present. A user's instruction in the first turn might get "forgotten" or lose influence as the conversation history grows, but the system prompt is a constant reminder of the AI's core persona and directives.

-----

## 3.&nbsp; Creating an Interactive Chat Loop

Now we'll modify our engine to support a back-and-forth conversation. To properly understand how chat applications work, we'll first build the interactive loop manually using basic Python.

### 3.1 The Manual Method: Building a `while` Loop

1.  Open `src/engine.py` in the VSCode editor.
2.  Replace the file's content with the code below. This script creates a `SimpleChatEngine` and a `while` loop that continuously prompts you for input.

In [ ]:
from llama_index.core.llms import ChatResponse
from llama_index.core.chat_engine import SimpleChatEngine
from llama_index.llms.groq import Groq

from src.config import LLM_SYSTEM_PROMPT
from src.model_loader import initialise_llm


def main_chat_loop() -> None:
    """Main chat loop for a conversational chatbot."""
    llm: Groq = initialise_llm()

    # Create a chat engine using the system prompt
    conversation: SimpleChatEngine = SimpleChatEngine.from_defaults(
        llm=llm,
        system_prompt=LLM_SYSTEM_PROMPT
    )

    print("--- Chat Started ---")
    print("Type 'exit' or 'quit' to end the conversation.")
    print("--------------------")

    while True:
        user_input: str = input("\nYou: ").strip()

        if user_input.lower() in ["quit", "exit"]:
            print("\nBot: Goodbye!")
            break

        if not user_input:
            continue

        response: ChatResponse = conversation.chat(user_input)
        print(f"\nBot: {response}")


You might wonder how the `SimpleChatEngine` "remembers" the conversation without us writing any code to store the history.

Behind the scenes, the engine is maintaining a simple list of messages. Every time you call `conversation.chat()`, it does the following:

1.  It takes your new message.
2.  It appends it to the list of all previous user and AI messages.
3.  It sends the entire list to the LLM API.
4.  It gets the AI's response back and adds that to the list as well.

This is why the conversation feels natural. However, it also reveals a critical limitation: with every turn, the amount of text sent to the API grows. This can eventually become slow and exceed the model's context window. We'll explore a solution to this in the challenges section.

With the interactive loop in place, run the application from the VSCode terminal to start a conversation with your bot.

In [ ]:
python main.py

### 3.2 The Easy Way: Using `chat_repl()`

Now that you understand the mechanics of building a chat loop from scratch, let's use the convenient, built-in method that LlamaIndex provides. This function, `chat_repl()`, handles the entire interactive loop for you.

1.  Open `src/engine.py` in the VSCode editor again.
2.  Replace the file's content with the code below. Notice that the entire `while` loop has been replaced by a single line.

In [ ]:
from llama_index.core.chat_engine import SimpleChatEngine
from llama_index.llms.groq import Groq

from src.config import LLM_SYSTEM_PROMPT
from src.model_loader import initialise_llm


def main_chat_loop() -> None:
    """Main chat loop for a conversational chatbot."""
    llm: Groq = initialise_llm()

    conversation: SimpleChatEngine = SimpleChatEngine.from_defaults(
        llm=llm,
        system_prompt=LLM_SYSTEM_PROMPT
    )

    conversation.chat_repl()


The `chat_repl()` method stands for **R**ead-**E**val-**P**rint **L**oop. By using it, your code becomes cleaner and less error-prone, as you don't have to manage the state of the conversation loop yourself.

-----

## 4.&nbsp; Run the Final Application

With the final `chat_repl()` version in place, run the application from the VSCode terminal to start your conversation.

In [ ]:
python main.py

To exit the conversation, simply type `exit`.

-----

## 5.&nbsp; Challenges and Further Exploration 😀

Now that you have a conversational chatbot, it's time to experiment.

### 1. Master the System Prompt

The `LLM_SYSTEM_PROMPT` in `src/config.py` is your primary tool for controlling the chatbot's personality.

**Challenge:**

  * **Change the Persona:** Modify the system prompt to make the chatbot act like a grumpy pirate, a cheerful robot, or a world-weary detective. How does the tone and phrasing of its answers change?
  * **Add Constraints:** Add a rule to the prompt, such as "You must always answer in the form of a haiku" or "Never use the letter 'e' in your responses." See how well the model adheres to these constraints.

### 2. Solve the Context Window Problem

As we discussed, `SimpleChatEngine` can be inefficient for long conversations. A better approach is to use an engine that summarises the conversation as it goes.

**Your Next Step: Implementing `ChatSummaryMemoryBuffer`**

This engine is a specialized memory component in LlamaIndex designed to manage long conversations efficiently by iteratively summarizing older messages while keeping recent interactions in full text.

**How the Memory Works**

* The Token Limit: It is the threshold for the "sliding window" of full-text history. Once the conversation exceeds this limit, the system doesn't just cut off the old messages; it uses the provided llm to summarize them.

* Summarization Logic: All older messages that no longer fit in the "full text" buffer are compressed into a single, concise summary message. This summary is then prepended to the recent messages when they are sent back to the LLM for the next turn.

* Prompt Customization: By default, it uses a system prompt like: "Write a concise summary about the contents of this conversation.".

**How to Implement It:**

1.  Open `src/engine.py`.
2.  Add the import `from llama_index.core.memory import ChatSummaryMemoryBuffer` at the top.
3.  In `main_chat_loop`, add the code below:

**Note:** When using the `ChatSummaryMemoryBuffer`, the summarization step counts as a separate API call. If you notice any Rate Limit errors, consider increasing the token_limit slightly.

In [ ]:
def main_chat_loop():
    """Main chat loop for a conversational chatbot."""
    llm: Groq = initialise_llm()

    # This uses the LLM to condense old messages once the limit is reached.
    memory = ChatSummaryMemoryBuffer.from_defaults(
        chat_history=[],
        llm=llm,
        token_limit=CHAT_MEMORY_TOKEN_LIMIT # Define in your config.py and import into engine.py
    )

    conversation: SimpleChatEngine = SimpleChatEngine.from_defaults(
        llm=llm,
        memory=memory, # This replaces the default standard buffer
        system_prompt=LLM_SYSTEM_PROMPT
    )

    conversation.chat_repl()



**Challenge:** Run the application with this new engine. Does the conversation feel different? Now that you know the pattern for changing the engine, try exploring other memory types from the LlamaIndex documentation and applying them.

### 3. Combine Everything

The true power of this platform lies in combining different models, prompts, and memory types.

**Challenge:** Experiment with various combinations. Which setup works best for you and why? Consider the trade-offs between response quality, speed, and conversational coherence. Try using the both the default memory and the `ChatSummaryMemoryBuffer` as well as
 various prompts. Note your results, then experiment again and make more notes.